# A first look at benchmark results

Run an experiment manifest, print its table, look at the difficulty proxies and draw two figures.

It runs the `smoke` manifest, which takes a second. Change `MANIFEST` to `corpus`, `tolerance` or `factorial` for the real experiments; those take minutes.

Needs the `analysis` extra for the figures, and the `validate` and `cp-sat` extras for the runs. Start Jupyter from the repository root, or from `notebooks/`.

In [ ]:
import tempfile
from pathlib import Path

from maxplus_solvers.experiments import benchmark, plotting, proxies, report
from maxplus_solvers.experiments.cli import proxy_table
from maxplus_solvers.experiments.manifest import load

ROOT = Path.cwd() if (Path.cwd() / "benchmarks").is_dir() else Path.cwd().parent
FIXTURES = ROOT / "benchmarks" / "fixtures"
MANIFEST = "smoke"

## Run it

One record per run. Each carries the solver version, the instance hash, the seed, the tolerances and the configuration, so that it can be run again.

In [ ]:
manifest = load(ROOT / "benchmarks" / "manifests" / f"{MANIFEST}.toml")
out = Path(tempfile.mkdtemp()) / f"{MANIFEST}.jsonl"
records = benchmark.run(manifest, FIXTURES, out)
print(manifest.description.strip())
print(len(records), "records in", out)

In [ ]:
print(report.table(records))

## Difficulty proxies

Numbers read off each corpus puzzle without searching it, beside what the whole searches above cost (for the puzzles the manifest ran), and their rank correlations within each regime. With eight puzzles a regime, a coefficient describes eight points.

In [ ]:
table, correlations = proxy_table(FIXTURES, records)
print(table)
print(correlations)

## Figures

Nodes of the whole search against the contact tolerance (one point per puzzle unless the manifest varies the tolerance), and one proxy against the cost.

In [ ]:
figure = plotting.nodes_by_tolerance(records)
figure

In [ ]:
index = benchmark.read_index(FIXTURES)
whole = [r for r in records if r["solver"] == "rust.contact_dfs" and r["exhaustive"]]
searched = {r["fixture"]: r for r in whole}
rows = {}
for fixture, record in searched.items():
    instance = benchmark.load_instance(FIXTURES / f"{fixture}.json")
    rows[fixture] = {**proxies.static(instance).to_dict(), **proxies.dynamic(record)}
figure = plotting.proxy_against_cost(rows, "mean_degree")
figure